# Week 9: Raster & Remote Sensing

This notebook covers:
- Loading satellite imagery with Rasterio
- Clipping rasters to an area of interest
- Calculating change between two dates
- Zonal statistics

In [ ]:
# Setup (run first)
import sys
if 'google.colab' in sys.modules:
    !pip install geopandas rasterio rasterstats -q

import geopandas as gpd
import rasterio
from rasterio.mask import mask
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("Ready!")

## 1. Load data

Update paths to your raster files and area of interest.

In [ ]:
DATA = Path("../data/processed/week09")

aoi = gpd.read_file(DATA / "aoi.geojson")

print(f"AOI bounds: {aoi.total_bounds}")
aoi.plot()

## 2. Clip rasters to AOI

Crop the satellite images to your study area.

In [ ]:
def clip_raster(raster_path, shapes):
    with rasterio.open(raster_path) as src:
        out_image, out_transform = mask(src, shapes.geometry, crop=True)
        out_meta = src.meta.copy()
        out_meta.update({
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform,
        })
    return out_image, out_meta

before, before_meta = clip_raster(DATA / "sentinel_before.tif", aoi)
after, after_meta = clip_raster(DATA / "sentinel_after.tif", aoi)

print(f"Clipped shape: {before.shape}")

## 3. Calculate change

Compute difference between the two dates.

In [ ]:
before_band = before[0].astype(float)
after_band = after[0].astype(float)

change = after_band - before_band

print(f"Mean change: {np.nanmean(change):.3f}")
print(f"Min: {np.nanmin(change):.3f}, Max: {np.nanmax(change):.3f}")

## 4. Visualise change

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(before_band, cmap="gray")
axes[0].set_title("Before")
axes[0].axis("off")

axes[1].imshow(after_band, cmap="gray")
axes[1].set_title("After")
axes[1].axis("off")

im = axes[2].imshow(change, cmap="RdYlGn", vmin=-0.3, vmax=0.3)
axes[2].set_title("Change")
axes[2].axis("off")
plt.colorbar(im, ax=axes[2])

plt.tight_layout()
plt.show()

## 5. Zonal statistics

Summarise change values by zone (e.g., neighbourhoods).

In [ ]:
from rasterstats import zonal_stats

zones = gpd.read_file(DATA / "zones.geojson")

stats = zonal_stats(zones, change, affine=before_meta["transform"], 
                    stats=["mean", "min", "max"], nodata=np.nan)

zones["change_mean"] = [s["mean"] for s in stats]
zones[["name", "change_mean"]].head()

## 6. Map zonal results

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

zones.plot(column="change_mean", cmap="RdYlGn", legend=True, ax=ax)
ax.set_title("Mean Change by Zone")
ax.set_axis_off()
plt.show()

## 7. Export

In [ ]:
output = DATA / "outputs"
output.mkdir(exist_ok=True)

zones.to_file(output / "zones_change.gpkg", driver="GPKG")
print(f"Saved to {output}")

---

**Done!** You've completed raster change detection in Python.